# 03 - Robustness: does any conclusion hinge on a choice?

**Presentation layer only.** Every number below is read from tables `run_all.py` wrote.
This notebook defines no analysis logic of its own. The command-line equivalent is:

```bash
python run_all.py --stage robustness   # ~30 seconds
python run_all.py --stage priors       # ~190 minutes, opt-in, feeds section 3
```

Four checks, each with a one-sentence verdict. They exist to answer a question the
headline results cannot answer about themselves: **which of these findings is a fact
about the data, and which is a fact about a decision someone made?**

| # | Check | Owed to |
|---|---|---|
| 1 | QLIKE ranking on the raw, unscaled Parkinson proxy | D10 |
| 2 | Refit cadence at 63 days instead of 21 | governing plan, Stage 6 |
| 3 | Prior sensitivity across the three `delta` candidates | D4 |
| 4 | Regime definition: trailing-volatility terciles instead of VIX bands | governing plan, Stage 5 |

The innovation ablation -- GARCH-normal against GARCH-t -- is a fifth, and it needs no
separate run: `garch_mle_normal` is carried through the same backtest and is already in
the Stage 4 tables. It appears in section 5.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import Image, display

from src import backtest as B

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def table(name: str) -> pd.DataFrame:
    return pd.read_csv(PROCESSED / f"{name}.csv")


HEADLINE = list(B.HEADLINE_MODELS)


## 1. Does the QLIKE ranking depend on the proxy scale constant?

Every headline point loss is scored against the Parkinson series multiplied by the frozen
constant `c = 1.517318`, which puts the intraday proxy on the close-to-close scale the
models forecast. `c` was estimated on the warm-up window and frozen before any
out-of-sample number existed -- but it is still an estimate, and D10 owes a demonstration
that the *ranking* does not depend on it.


In [ ]:
losses = table("eval_raw_proxy_losses")
qlike = losses[losses["loss"] == "qlike"]
qlike.pivot(index="model", columns="sample", values="mean").reindex(HEADLINE).round(4)


In [ ]:
qlike.sort_values(["sample", "mean"])[
    ["sample", "model", "n", "mean", "ci_lower", "ci_upper"]
].round(4)


**Verdict: the ranking is unchanged.** `garch_bayes` < `garch_mle` < `ewma` < `yesterday`
under both proxies, so the ordering is a fact about the models rather than about `c`.

Two details worth carrying into the report. The *levels* are not comparable across the two
columns -- the raw proxy is systematically smaller than the target the models forecast, so
every model is penalised for over-prediction against it, which is exactly the bias `c`
exists to remove. And the gap between the two baselines closes substantially on the raw
proxy, to the point where their bootstrap intervals overlap heavily: the raw-proxy
comparison can separate GARCH from no-GARCH but not `ewma` from `yesterday`.


## 2. Does anything hinge on refitting every 21 days?

The locked design refits every 21 trading days. This re-runs the **frequentist track** at
63 and compares. Frequentist-only is a stated choice, not an oversight: the question is
whether conclusions depend on how often parameters are re-estimated, and the frequentist
track answers it in twenty seconds where the Bayesian track would cost another ninety-five
minutes.


In [ ]:
table("eval_cadence_comparison").round(4)


**Verdict: nothing hinges on it.** `garch_mle`'s mean QLIKE moves from 0.4610 to 0.4616
and its 99% VaR breaches from 37 to 35 -- Kupiec still rejects at both cadences, and the
GARCH-normal ablation still fails far worse than the t at both. No conclusion in the
report changes.

The two baselines are bit-identical at both cadences, as they must be: neither estimates
a parameter, so the refit cadence is a genuine no-op for them. `--stage robustness`
asserts that on every run and stops if it is ever false, because a cadence that reached a
parameter-free model would be a harness bug rather than a robustness finding.


## 3. How much of the interval result is the `delta` prior?

The most load-bearing check in this notebook. `research_log.md` §1.13 found that the
frequentist-Bayesian interval difference is mostly **the priors moving the point
estimate**, not parameter uncertainty -- so the obvious next question is how much of *that*
is specific to `Beta(3, 1)`, the candidate D4 froze.

Answering it costs two more full Bayesian backtests, one per rejected candidate, at the
production config and the same `mcmc_seed` so the prior is the only thing that differs.
They run from `--stage priors` and write to `data/processed/prior_sensitivity/`.


In [ ]:
try:
    display(table("eval_prior_sensitivity").round(5))
except FileNotFoundError:
    print("Not yet run. `python run_all.py --stage priors` (~190 minutes), then")
    print("`python run_all.py --stage robustness` to build the table.")


The prior does move the posterior, and in the direction §1.10's structural argument
predicted. Measured on the warm-up window before the production runs began, the 99th
percentile of `alpha + beta` is 0.9969 under the frozen `Beta(3, 1)`, 0.9845 under
`Beta(10, 2)` and 0.9998 under `Beta(1, 1)` -- so the candidates separate most visibly at
the stationarity boundary, which is where §1.9 recorded the data pressing.

**If this check is cut for budget, it must be cut explicitly into the limitations section
rather than by omission.** An unexamined prior behind a finding that is *about* priors is
the one gap this project cannot leave silent.


## 4. Does the regime finding depend on the VIX thresholds?

Stage 5 found that the GARCH models' 99% VaR is well calibrated in calm and in stress and
fails in the middle band. The regime labels come from the VIX close at `t-1`, binned at
the locked thresholds of 15 and 25. This repeats the split on terciles of trailing 21-day
Parkinson volatility.

**The cut points are estimated on the warm-up window alone.** Terciles over the full
sample would make every label a function of days that had not happened yet -- the crisis
regime defined using the crisis. See `docs/problems-and-solutions.md` #43.


In [ ]:
trailing = table("eval_regime_var_trailing")
vix = table("eval_regime_var")
both = pd.concat([vix, trailing])
both[both["model"].isin(HEADLINE)].pivot(
    index=["model", "regime"], columns="sample", values="rate"
).round(4)


In [ ]:
trailing[trailing["model"].isin(HEADLINE)][
    ["model", "regime", "n", "breaches", "rate", "ci_lower", "ci_upper", "kupiec_p"]
].round(4)


**Verdict: the finding survives, and the two definitions disagree about the details.**
Under terciles the GARCH models are closest to nominal in the *top* bucket (1.35%, Kupiec
p = 0.29) and worst in the *bottom* one (2.06%, p = 0.012); under the VIX bands the weak
spot is the middle. They agree on what matters: **these models are not worse in stress,
they are worse outside it.**

The two are different partitions of the same days and **no row compares across them**. The
warm-up was calm, so the tercile "stressed" bucket holds 1,040 of the 2,134 evaluation
days against the VIX definition's 347 -- a far weaker notion of stress. That asymmetry is
the price of estimating the thresholds without look-ahead, and it is reported rather than
corrected, because every available correction reintroduces the look-ahead.


## 5. The innovation ablation: normal against Student-t

The cheap, decisive lever on 99% tail coverage, and it needs no extra run --
`garch_mle_normal` is carried through the same backtest as an ablation (D15) and is
already in the Stage 4 tables. It is filtered out of every headline table and reported
only here.


In [ ]:
pit = table("eval_pit").query("sample == 'own days'")
var_backtests = table("eval_var_backtests").query("sample == 'own days'")
comparison = pit.merge(var_backtests, on=["sample", "model", "n"])
comparison[comparison["model"].isin(["garch_mle", "garch_mle_normal"])][
    ["model", "n", "ks_stat", "ks_p", "breaches", "rate", "kupiec_p", "independence_p"]
].round(4)


**Verdict: the Student-t innovation is what buys the tail, and it is decisive.** The
normal-innovation variant fails the PIT uniformity test at p = 2.6e-4 where the t passes
at 0.115, and takes 53 breaches of its 99% VaR against the t's 37 and an expected 21. The
two share everything else -- the same likelihood shape, the same estimator, the same
harness -- so the difference is the innovation distribution alone.

Set beside the project's central measurement, this is the sharpest thing in the notebook:
choosing the innovation distribution moves 99% tail coverage decisively, while integrating
over parameter uncertainty moves it not at all.


## What the robustness checks do and do not license

**Do.** The QLIKE ranking, the regime conclusion and the VaR verdicts survive every
alternative tried here: a different proxy scale, a different refit cadence, and a
different regime definition estimated without look-ahead.

**Do not.** None of these checks addresses the design's structural limits, which belong in
the report's limitations section and are not robustness questions at all: one asset, one
horizon, roughly two stress episodes, one GARCH family, and a proxy that is biased for the
quantity being forecast. A conclusion can be robust to every knob tested and still be
specific to SPY between 2017 and 2025.
